# Exploratory Data Analysis (EDA)
## Hazara Division Air Quality Intelligence System

This notebook performs a comprehensive analysis of the air quality data collected for the Hazara Division, KPK.

**Objectives:**
- Understand the distribution of pollutants across districts
- Identify temporal patterns (hourly, daily, monthly)
- Analyze correlations between pollutants and the AQI
- Visualize district-wise AQI comparisons

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.database import AQIDatabase

# Style
plt.style.use('dark_background')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

## 1. Load Data from MongoDB

In [ ]:
# Connect to MongoDB and fetch all data
db = AQIDatabase()
df = db.fetch_data()

print(f'Total records: {len(df)}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Districts: {df["district"].unique().tolist()}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(10)

## 2. Data Overview & Missing Values

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values!')

print(f'\n=== Statistical Summary ===')
df.describe().round(2)

## 3. AQI Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall AQI histogram
axes[0].hist(df['us_aqi'].dropna(), bins=50, color='#10B981', edgecolor='white', alpha=0.8)
axes[0].set_title('AQI Distribution (All Districts)', fontweight='bold')
axes[0].set_xlabel('US AQI')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['us_aqi'].mean(), color='#ef4444', linestyle='--', label=f'Mean: {df["us_aqi"].mean():.1f}')
axes[0].legend()

# AQI boxplot by district
district_data = [df[df['district'] == d]['us_aqi'].dropna() for d in df['district'].unique()]
bp = axes[1].boxplot(district_data, labels=df['district'].unique(), patch_artist=True)
colors = ['#10B981', '#3B82F6', '#F59E0B', '#EF4444', '#8B5CF6', '#EC4899']
for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('AQI by District', fontweight='bold')
axes[1].set_ylabel('US AQI')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Pollutant Correlation Matrix

In [ ]:
# Select numeric pollutant columns
pollutant_cols = ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'us_aqi']
corr_matrix = df[pollutant_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn_r', center=0, square=True,
    linewidths=0.5, ax=ax,
    vmin=-1, vmax=1
)
ax.set_title('Pollutant Correlation Matrix\n(Hazara Division)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('\nCorrelation with US AQI:')
print(corr_matrix['us_aqi'].sort_values(ascending=False).round(3))

## 5. Temporal Patterns

In [ ]:
# Ensure date is datetime
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Hourly AQI Pattern
hourly_avg = df.groupby('hour')['us_aqi'].mean()
axes[0].bar(hourly_avg.index, hourly_avg.values, color='#10B981', alpha=0.8, edgecolor='white')
axes[0].set_title('Average AQI by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Average AQI')
axes[0].set_xticks(range(0, 24, 3))

# Daily AQI Pattern
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_avg = df.groupby('day_of_week')['us_aqi'].mean().reindex(day_order)
axes[1].bar(range(7), daily_avg.values, color='#3B82F6', alpha=0.8, edgecolor='white')
axes[1].set_title('Average AQI by Day of Week', fontweight='bold')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1].set_ylabel('Average AQI')

# Monthly AQI Pattern
monthly_avg = df.groupby('month')['us_aqi'].mean()
axes[2].plot(monthly_avg.index, monthly_avg.values, 'o-', color='#F59E0B', linewidth=2, markersize=8)
axes[2].set_title('Average AQI by Month', fontweight='bold')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average AQI')
axes[2].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

## 6. District-wise Time Series Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

districts = df['district'].unique()
colors = ['#10B981', '#3B82F6', '#F59E0B', '#EF4444', '#8B5CF6', '#EC4899']

for i, district in enumerate(districts):
    district_df = df[df['district'] == district].copy()
    # Resample to daily average for cleaner visualization
    district_df = district_df.set_index('date').resample('D')['us_aqi'].mean().reset_index()
    ax.plot(
        district_df['date'], district_df['us_aqi'],
        label=district, color=colors[i % len(colors)],
        alpha=0.8, linewidth=1.5
    )

ax.set_title('Daily Average AQI Across Hazara Division Districts', fontweight='bold', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('US AQI (Daily Average)')
ax.legend(loc='upper left', framealpha=0.5)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 7. PM2.5 vs AQI Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, district in enumerate(districts):
    d = df[df['district'] == district]
    ax.scatter(
        d['pm2_5'], d['us_aqi'],
        alpha=0.3, s=10, label=district,
        color=colors[i % len(colors)]
    )

ax.set_title('PM2.5 vs US AQI (by District)', fontweight='bold')
ax.set_xlabel('PM2.5 (µg/m³)')
ax.set_ylabel('US AQI')
ax.legend(markerscale=3)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f'\nPM2.5 to AQI Correlation: {df["pm2_5"].corr(df["us_aqi"]):.3f}')

## 8. Key Findings Summary

After running this notebook with live data, document your key findings here:

1. **Dominant Pollutant**: PM2.5 is expected to show the strongest correlation with AQI
2. **Temporal Patterns**: Look for rush-hour peaks (morning/evening) and seasonal variations
3. **District Differences**: Abbottabad and Haripur (more urbanized) may show higher AQI
4. **Data Quality**: Note any gaps or anomalies in the data collection